# 🏆 Synthèse Finale : Sélection du Meilleur Modèle

Ce notebook agrège tous les résultats obtenus lors des différentes phases du projet (Baselines, Hybrides, Embeddings, Deep Learning) pour désigner officiellement le modèle champion.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


sns.set_theme(style="whitegrid")
OUTPUT_DIR = Path("../outputs")

## 1. Collecte des Résultats
Nous scannons récursivement le dossier `outputs` pour récupérer tous les fichiers `.csv` de métriques.

In [ ]:
files = list(OUTPUT_DIR.glob("**/NB*_results.csv")) + \
        list(OUTPUT_DIR.glob("**/*tuned_results.csv")) + \
        list(OUTPUT_DIR.glob("**/*baseline_results.csv"))

all_results = []
for file in files:
    try:
        df = pd.read_csv(file)
        cols = ['pipeline', 'test_f1_class_1', 'test_recall_class_1', 'test_precision_class_1', 'test_accuracy']
        existing = [c for c in cols if c in df.columns]
        all_results.append(df[existing])
    except Exception:
        pass

results_df = pd.concat(all_results, ignore_index=True).drop_duplicates(subset=['pipeline'])
results_df['pipeline'] = results_df['pipeline'].replace({'test': 'DistilBERT Tuned (Champion)'})
results_df = results_df.sort_values('test_f1_class_1', ascending=False).reset_index(drop=True)

print(f"Total de modèles comparés : {len(results_df)}")
results_df.head(10)

## 2. Visualisation des Performances
Comparaison du **F1-Score (Classe 1)** pour le Top 10 des modèles.

In [ ]:
plt.figure(figsize=(12, 6))
top_10 = results_df.head(10)
ax = sns.barplot(data=top_10, x='test_f1_class_1', y='pipeline', palette='viridis')

plt.title('Top 10 Modèles - F1-Score sur la classe Disaster', fontsize=15, fontweight='bold')
plt.xlabel('F1-Score (Test Set)', fontsize=12)
plt.ylabel('Modèle / Pipeline', fontsize=12)
plt.xlim(0.6, 0.8)

# Ajout des valeurs sur les barres
for p in ax.patches:
    ax.annotate(f'{p.get_width():.4f}', (p.get_width() + 0.005, p.get_y() + p.get_height()/2), va='center')

plt.show()

## 3. Analyse du Compromis Précision/Rappel
Le modèle champion doit offrir le meilleur équilibre.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(results_df['test_recall_class_1'], results_df['test_precision_class_1'],
            alpha=0.6, s=100, c=results_df['test_f1_class_1'], cmap='coolwarm')

plt.title('Précision vs Rappel (Toutes expérimentations)')
plt.xlabel('Rappel (Recall)')
plt.ylabel('Précision')
plt.grid(True)

# Annotation du champion
champion = results_df.iloc[0]
plt.annotate(champion['pipeline'], (champion['test_recall_class_1'], champion['test_precision_class_1']),
             xytext=(10, 10), textcoords='offset points', fontweight='bold', color='darkred')

plt.colorbar(label='F1-Score')
plt.show()

## 4. Conclusion Finale

Le modèle **DistilBERT Tuned** est officiellement désigné comme le meilleur modèle pour la détection de tweets de catastrophes.

**Points clés :**
- **Stabilité :** Il offre une précision de ~75% et un rappel de ~80%.
- **Supériorité :** Il surpasse les approches classiques (TF-IDF) de plus de 7 points de F1-Score.
- **Robustesse :** C'est le modèle déployé en production sur notre API FastAPI.